In [1]:
import numpy as np
import pandas as pd
from zebrafish_ms2_paper.utils import pboc_rc, style_axes, colors, fontsize
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.special import gamma, hyp1f1, loggamma
from scipy.optimize import minimize
import pickle

In [2]:
%matplotlib qt

In [3]:
mpl.rcParams.update(pboc_rc)
mpl.rcParams['pdf.fonttype'] = 42

In [4]:
def two_state_prob_dist(m, k_on, k_off, r, γ):
    return (
            gamma(k_on / γ + m) 
            * gamma(k_on / γ + k_off / γ) 
            / gamma(m + 1) 
            / gamma(k_on / γ + k_off / γ + m) 
            / gamma(k_on / γ)
           ) * (r / γ) ** m * hyp1f1(k_on / γ + m, k_on / γ + k_off / γ + m, -r / γ)

def two_state_log_likelihood(m, k_on, k_off, r, γ):
    return (
            loggamma(k_on / γ + m)
            + loggamma(k_on / γ + k_off / γ) 
            - loggamma(m + 1) 
            - loggamma(k_on / γ + k_off / γ + m) 
            - loggamma(k_on / γ)
            + m * np.log(r / γ)
            + np.log(hyp1f1(k_on / γ + m, k_on / γ + k_off / γ + m, -r / γ))
            )


def func(p, m_arr, γ):
    k_on, k_off, r = p
    #return -np.sum(np.log10(two_state_prob_dist(m_arr, k_on, k_off, r, γ)))
    return -np.sum(two_state_log_likelihood(m_arr, k_on, k_off, r, γ))


def initial_param_guess():
    k_on = 0.5
    k_off = 10.0
    r = 100.0
    return k_on, k_off, r

def fit_count_data(m_arr, γ):
    p0 = initial_param_guess()
    
    res = minimize(func, p0, args=(m_arr, γ), bounds=((0.01, 10.0), (0.01, 100.0), (1.0, 1000.0)))
    
    return res
    

In [5]:
df_dict = pd.read_excel(r'/home/brandon/Downloads/41586_2020_3055_MOESM7_ESM.xlsx', sheet_name=None)

In [6]:
df = df_dict['1']
sheet_numbers = np.arange(2, len(df_dict) + 1)
for n in sheet_numbers:
    df = pd.concat((df, df_dict[f'{n}']), axis=0)

In [8]:
her1 = df.get('her1.1').values
#her1 = df.her1.values
her1 = her1[~np.isnan(her1)]

#her1_2 = df.get('her1.1').values
#her1_2 = her1_2[~np.isnan(her1_2)]

#her1 = np.concatenate((her1, her1_2))

In [14]:
"""each embryo individually"""
%matplotlib qt

fig, axs = plt.subplots(5, 5, figsize=(10, 10))
γ = 0.23
param_arr = np.zeros((len(df_dict), 3))
counter = 0
for i in range(5):
    for j in range(5):
        if counter > len(df_dict) + 1:
            ax = style_axes(axes[i, j], fontsize=9)
            continue
            
        counter += 1
        print(f'{counter} of {len(sheet_numbers) + 1}')
        this_df = df_dict[f'{counter}']
        this_her1 = this_df.her1.values
        this_her1 = this_her1[~np.isnan(this_her1)]

        this_her1_2 = this_df.get('her1.1').values
        this_her1_2 = this_her1_2[~np.isnan(this_her1_2)]

        #this_her1 = np.concatenate((this_her1, this_her1_2))
        #this_her1 = this_her1_2
        this_her1 = this_her1
        
        res = fit_count_data(this_her1, γ)

        param_arr[counter - 1] = res.x

        bins = np.arange(100)
        counts, _ = np.histogram(this_her1, bins)
        probs = counts / np.sum(counts)

        k_on, k_off, r = res.x
        fit_prob_dist = np.exp(two_state_log_likelihood(bins, k_on, k_off, r, γ))
        ax = axs[i, j]
        ax.plot(bins, fit_prob_dist, '-', color=colors['blue'], linewidth=2, alpha=1, label='Two-state fit')
        ax.plot(bins[:-1], probs, 'ko', markersize=3, label='smFISH data')
        #ax.set_xlabel("$her1$ mRNA count", fontsize=fontsize)
        #ax.set_ylabel('probability density ', fontsize=fontsize)
        ax = style_axes(ax, fontsize=9)
        
fig.tight_layout()
    

1 of 23
2 of 23
3 of 23
4 of 23
5 of 23
6 of 23
7 of 23
8 of 23
9 of 23
10 of 23
11 of 23
12 of 23
13 of 23
14 of 23
15 of 23
16 of 23
17 of 23
18 of 23
19 of 23
20 of 23
21 of 23
22 of 23
23 of 23
24 of 23


KeyError: '24'

In [219]:
param_arr

array([[  0.54688211,  12.36065665,  99.75181213],
       [  0.33787959,   1.05685005,  34.676012  ],
       [  0.3792936 ,   1.6320285 ,  35.76317801],
       [  0.46281211,   6.19334899, 107.40375757],
       [  0.53257214,   7.78817562, 101.44408693],
       [  0.45958351,   9.5437661 , 100.04598503],
       [  0.57802289,  10.35066809,  99.92384407],
       [  0.47558368,  15.11652054,  98.89485347],
       [  0.45840625,   7.76116987, 101.47896696],
       [  0.47465316,   9.18286268, 100.12811487],
       [  0.62425826,   7.78244906, 102.5754203 ],
       [  0.41775071,   5.506616  , 106.26843202],
       [  0.50039468,   7.69176615, 101.66479499],
       [  0.41176874,   7.09737746, 102.78399735],
       [  0.39109658,   7.27414965, 100.68162728],
       [  0.53625995,  10.6206648 ,  99.92792   ],
       [  0.50436955,   7.33699325, 101.51546459],
       [  0.52283654,  18.61769517,  98.66581784],
       [  0.52589749,  10.31624403,  99.96493415],
       [  0.38250132,  11.10086

In [220]:
np.mean(param_arr, axis=0)

array([ 0.47223716,  8.68990068, 95.75299845])

In [221]:
np.std(param_arr, axis=0)

array([ 0.06911084,  3.85765941, 18.8178123 ])

In [216]:
with open(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/data/ozbudak_fits/2025_09_18_ozbudak_fits.pkl', 'rb') as file:
    param_arr_og = pickle.load(file)
param_arr_og

array([[9.54756647e-01, 2.12284754e+01, 9.87847458e+01],
       [8.40010480e-01, 9.27175641e+00, 1.00875961e+02],
       [7.57228053e-01, 9.39016017e+00, 1.00703113e+02],
       [8.67091662e-01, 1.07293458e+01, 9.95368638e+01],
       [8.60160813e-01, 1.37492192e+01, 9.91125999e+01],
       [9.73354072e-01, 2.15235951e+01, 9.87528088e+01],
       [1.04690974e+00, 1.72719767e+01, 9.84769315e+01],
       [9.55300662e-01, 3.03163509e+01, 9.74270118e+01],
       [8.01332115e-01, 1.15259581e+01, 9.91801938e+01],
       [1.02198470e+00, 2.15010638e+01, 9.87272856e+01],
       [8.67088137e-01, 4.55474725e+00, 5.04900936e+01],
       [8.56777114e-01, 8.90756989e+00, 1.01424081e+02],
       [7.85801970e-01, 1.08264202e+01, 9.96308125e+01],
       [6.91143752e-01, 1.01467602e+01, 9.99898008e+01],
       [8.21291105e-01, 1.33103099e+01, 9.89495132e+01],
       [1.06224286e+00, 1.94334513e+01, 9.89528241e+01],
       [1.00173510e+00, 1.00000000e+02, 6.63895192e+02],
       [9.02520340e-01, 3.76852

In [217]:
with open(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/data/ozbudak_fits/2025_09_18_ozbudak_fits.pkl', 'wb') as file:
    pickle.dump(param_arr, file)

In [175]:
np.mean(param_arr, axis=0)

array([  0.86423383,  19.26372035, 121.71858586])

In [176]:
np.std(param_arr, axis=0)

array([  0.12091612,  18.72904371, 116.02316355])

## Plot combined distributions

In [15]:
"""plot mean + std of experimetnal dists and mean + std of fits"""
#with open(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/data/ozbudak_fits/2025_09_18_ozbudak_fits.pkl', 'rb') as file:
#    param_arr = pickle.load(file)
    
γ = 0.23
fontsize = 11
data_bins = np.arange(0, 100, 2)
data_prob_dists = np.zeros((len(df_dict), len(data_bins) - 1))

fit_bins = np.arange(100)
fit_prob_dists =  np.zeros((len(df_dict), len(fit_bins)))
data_color = colors['purple']
fit_color = colors['green']
for i in range(len(df_dict)):
    this_df = df_dict[f'{i + 1}']
    this_her1 = this_df.her1.values
    this_her1 = this_her1[~np.isnan(this_her1)]

    this_her1_2 = this_df.get('her1.1').values
    this_her1_2 = this_her1_2[~np.isnan(this_her1_2)]

    #this_her1 = np.concatenate((this_her1, this_her1_2))
    #this_her1 = this_her1_2
    this_her1 = this_her1
  
    counts, _ = np.histogram(this_her1, data_bins)
    data_prob_dists[i] = counts / np.sum(counts) / np.diff(data_bins)[0]
    
    # fit
    k_on, k_off, r = param_arr[i]
    fit_prob_dists[i] = np.exp(two_state_log_likelihood(fit_bins, k_on, k_off, r, γ))
    

mean_data_prob_dists = np.mean(data_prob_dists, axis=0)
std_data_prob_dists = np.std(data_prob_dists, axis=0)

mean_fit_prob_dists = np.mean(fit_prob_dists, axis=0)
std_fit_prob_dists = np.std(fit_prob_dists, axis=0)

plt.figure(figsize=(5, 4))
plt.errorbar(data_bins[:-1], mean_data_prob_dists, std_data_prob_dists, marker='o', markeredgecolor=data_color, markersize=3, markerfacecolor='none', ecolor=data_color, color='none', label='smFISH data, mean $\pm$ std. dev\nacross 23 embryos')
#plt.fill_between(bins[:-1], mean_fit_prob_dists - std_fit_prob_dists, mean_fit_prob_dists + std_fit_prob_dists, color=fit_color, alpha=1.0, label='Two-state\nmodel fit')
plt.plot(fit_bins, mean_fit_prob_dists, color=fit_color, alpha=1.0, linewidth=2, label='Two-state bursting model fit', zorder=100)

#plt.plot(bins[:-1], mean_fit_prob_dists - std_fit_prob_dists, color=fit_color, alpha=1.0, label='Two-state\nmodel fit')
#plt.plot(bins[:-1], mean_fit_prob_dists + std_fit_prob_dists, color=fit_color, alpha=1.0, label='_none_')

plt.xlabel("$her1$ mRNA count", fontsize=fontsize)
plt.ylabel('probability density ', fontsize=fontsize)
plt.legend(fontsize=fontsize, facecolor='w')
ax = style_axes(plt.gca(), fontsize=fontsize)
plt.tight_layout()

<>:41: SyntaxWarning: invalid escape sequence '\p'
<>:41: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_264789/3310862360.py:41: SyntaxWarning: invalid escape sequence '\p'
  plt.errorbar(data_bins[:-1], mean_data_prob_dists, std_data_prob_dists, marker='o', markeredgecolor=data_color, markersize=3, markerfacecolor='none', ecolor=data_color, color='none', label='smFISH data, mean $\pm$ std. dev\nacross 23 embryos')


In [284]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/ozbudak_plots/ozbudak_combined_dist.pdf')

## Plot each embryo's fit individually

In [11]:
# with open(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/data/ozbudak_fits/2025_09_18_ozbudak_fits.pkl', 'rb') as file:
#     param_arr = pickle.load(file)
    
fig, axs = plt.subplots(3, 8, figsize=(10.5, 4.5))
γ = 0.23
data_color = colors['purple']
fit_color = colors['green']
counter = 0
for i in range(3):
    for j in range(8):
        if counter >= len(df_dict):
            ax = axs[i, j]
            ax.set_xticks([0, 50, 100])
            ax.set_yticks([0, 0.05], labels=[])
            ax.set_ylim([0, 0.07])

            ax = style_axes(ax, fontsize=fontsize)
            continue
        
#         if i == 4 and j == 0:
#             ax = axs[i, j]
#             ax.set_xticks([])
#             ax.set_yticks([])
#             ax = style_axes(ax, fontsize=fontsize)
#             continue
            
        counter += 1
        this_df = df_dict[f'{counter}']
        this_her1 = this_df.her1.values
        this_her1 = this_her1[~np.isnan(this_her1)]

        this_her1_2 = this_df.get('her1.1').values
        this_her1_2 = this_her1_2[~np.isnan(this_her1_2)]

        #this_her1 = np.concatenate((this_her1, this_her1_2))
        this_her1 = this_her1_2

        k_on, k_off, r = param_arr[counter - 1]

        bins = np.arange(100)
        counts, _ = np.histogram(this_her1, bins)
        probs = counts / np.sum(counts)

        fit_prob_dist = np.exp(two_state_log_likelihood(bins, k_on, k_off, r, γ))
        ax = axs[i, j]
        ax.plot(bins[:-1], probs, 'o', markersize=3, markeredgecolor=data_color, markerfacecolor='none', label='smFISH data')
        ax.plot(bins, fit_prob_dist, '-', color=fit_color, linewidth=2, alpha=1, label='Two-state fit')

        #ax.set_xlabel("$her1$ mRNA count", fontsize=fontsize)
        #ax.set_ylabel('probability density ', fontsize=fontsize)
        ax.set_ylim([0, 0.07])
        
        if j == 0:
            ax.set_yticks([0, 0.05])
        else:
            ax.set_yticks([0, 0.05], labels=[])
        
        if i == 2:
            ax.set_xticks([0, 50, 100])
        else:
            ax.set_xticks([0, 50, 100], labels=[])

        if j == 0 and i == 1:
            ax.set_ylabel('probability density', fontsize=fontsize)
        if i == 2 and j == 3:
            ax.set_xlabel("$her1$ mRNA count", fontsize=fontsize)
        
        ax = style_axes(ax, fontsize=fontsize)
        
#fig.tight_layout()

In [293]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/ozbudak_plots/ozbudak_all_dists_3x8.pdf')


In [286]:
with open(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/data/ozbudak_fits/2025_09_18_ozbudak_fits.pkl', 'rb') as file:
    param_arr = pickle.load(file)
    
fig, axs = plt.subplots(4, 6, figsize=(7.5, 5.5))
γ = 0.23
data_color = colors['purple']
fit_color = colors['green']
counter = 0
for i in range(4):
    for j in range(6):
        if counter >= len(df_dict):
            ax = axs[i, j]
            ax.set_xticks([])
            ax.set_yticks([])
            ax = style_axes(ax, fontsize=fontsize)
            continue
        
#         if i == 4 and j == 0:
#             ax = axs[i, j]
#             ax.set_xticks([])
#             ax.set_yticks([])
#             ax = style_axes(ax, fontsize=fontsize)
#             continue
            
        counter += 1
        this_df = df_dict[f'{counter}']
        this_her1 = this_df.her1.values
        this_her1 = this_her1[~np.isnan(this_her1)]

        this_her1_2 = this_df.get('her1.1').values
        this_her1_2 = this_her1_2[~np.isnan(this_her1_2)]

        this_her1 = np.concatenate((this_her1, this_her1_2))


        k_on, k_off, r = param_arr[counter - 1]

        bins = np.arange(100)
        counts, _ = np.histogram(this_her1, bins)
        probs = counts / np.sum(counts)

        fit_prob_dist = np.exp(two_state_log_likelihood(bins, k_on, k_off, r, γ))
        ax = axs[i, j]
        ax.plot(bins[:-1], probs, 'o', markersize=3, markeredgecolor=data_color, markerfacecolor='none', label='smFISH data')
        ax.plot(bins, fit_prob_dist, '-', color=fit_color, linewidth=2, alpha=1, label='Two-state fit')

        #ax.set_xlabel("$her1$ mRNA count", fontsize=fontsize)
        #ax.set_ylabel('probability density ', fontsize=fontsize)
        ax.set_ylim([0, 0.07])
        
        if j == 0:
            ax.set_yticks([0, 0.05])
        else:
            ax.set_yticks([0, 0.05], labels=[])
        
        if i == 3:
            ax.set_xticks([0, 50, 100])
        else:
            ax.set_xticks([0, 50, 100], labels=[])

        if j == 0 and i == 2:
            ax.set_ylabel('probability density', fontsize=fontsize)
        if i == 3 and j == 2:
            ax.set_xlabel("$her1$ mRNA count", fontsize=fontsize)
        
        ax = style_axes(ax, fontsize=fontsize)
        
#fig.tight_layout()

In [287]:
plt.savefig(r'/home/brandon/Documents/Code/zebrafish-ms2-paper/figures/ozbudak_plots/ozbudak_all_dists_4x6.pdf')
    

## Visualize the raw data

In [158]:
import napari

In [205]:
"""napari test"""
# collect her1 points for embryo 1
df = df_dict['2']
points_1 = df.get(["Cell Position Z", "Cell Position Y", "Cell Position X"]).dropna().values
vals = df.get("her1").dropna().values
#points_2 = df.get(["Cell Position Z.1", "Cell Position Y.1", "Cell Position X.1"]).dropna().values

viewer = napari.Viewer()
big_value = 80
for i in range(len(df_dict) - 1):
    df = df_dict.get(f'{i+1}')
    points_1 = df.get(["Cell Position Z", "Cell Position Y", "Cell Position X"]).dropna().values
    vals = df.get("her1").dropna().values
    viewer.add_points(points_1)
    adjusted_vals = vals / big_value
    adjusted_vals[adjusted_vals > 1] = 1
    colors = np.concatenate((np.zeros((len(vals), 1)), np.expand_dims(adjusted_vals, axis=1), np.zeros((len(vals), 1)), np.ones((len(vals), 1))), axis=1)
    viewer.layers[-1].face_color = colors
    viewer.layers[-1].border_color = colors
#viewer.add_points(points_2)
#colors = np.concatenate((np.zeros((len(vals), 1)), np.expand_dims(vals / 80, axis=1), np.zeros((len(vals), 1)), np.ones((len(vals), 1))), axis=1)
for layer in viewer.layers:
    layer.scale = (0.27, 0.13, 0.13)
    #layer.face_color = colors
    #layer.border_color = colors
    layer.opacity = 0.75

In [232]:
viewer = napari.Viewer()
i = 0
df = df_dict.get(f'{i+1}')
points_1 = df.get(["Cell Position Z", "Cell Position Y", "Cell Position X"]).dropna().values
vals = df.get("her1").dropna().values
viewer.add_points(points_1)


adjusted_vals = vals / big_value
adjusted_vals[adjusted_vals > 1] = 1
colors = np.concatenate((np.zeros((len(vals), 1)), np.expand_dims(adjusted_vals, axis=1), np.zeros((len(vals), 1)), np.ones((len(vals), 1))), axis=1)
viewer.layers[-1].face_color = colors
viewer.layers[-1].border_color = colors

points_2 = df.get(["Cell Position Z.1", "Cell Position Y.1", "Cell Position X.1"]).dropna().values
z_drift = np.mean(points_1[:, 0]) - np.mean(points_2[:, 0])
points_2[:, 0] += z_drift
vals_2 = df.get("her1.1").dropna().values
viewer.add_points(points_2)
adjusted_vals = vals_2 / big_value
adjusted_vals[adjusted_vals > 1] = 1
colors_2 = np.concatenate((np.zeros((len(vals_2), 1)), np.expand_dims(adjusted_vals, axis=1), np.zeros((len(vals_2), 1)), np.ones((len(vals_2), 1))), axis=1)
viewer.layers[-1].face_color = colors_2
viewer.layers[-1].border_color = colors_2

for layer in viewer.layers:
    layer.scale = (0.27, 0.13, 0.13)
    #layer.face_color = colors
    #layer.border_color = colors
    layer.opacity = 0.75

Traceback (most recent call last):
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/napari/_vispy/camera.py", line 284, in viewbox_mouse_event
    super().viewbox_mouse_event(event)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/scene/cameras/perspective.py", line 238, in viewbox_mouse_event
    self._update_rotation(event)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/scene/cameras/arcball.py", line 64, in _update_rotation
    Quaternion(*_arcball(self._event_value, wh)) *
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/scene/cameras/arcball.py", line 100, in _arcball
    x, y = xy
    ^^^^
ValueError: too many values t

In [230]:
np.std(points_2[:, 0]) * 0.27

20.367283844503355

In [196]:
colors = np.concatenate((np.zeros((len(vals), 1)), np.expand_dims(vals / np.max(vals) * 255, axis=1), np.zeros((len(vals), 1)), np.ones((len(vals), 1))), axis=1)


In [226]:
df.keys()

Index(['3 ID', 'Cell Position X', 'Cell Position Y', 'Cell Position Z', 'her1',
       'her7', 'Cell Volum', '1 ID', 'Cell Position X.1', 'Cell Position Y.1',
       'Cell Position Z.1', 'her1.1', 'her7.1', 'Cell Volum.1'],
      dtype='object')

In [197]:
colors.shape

(945, 4)

In [190]:
vals.shape

(945,)

In [189]:
colors.shape

(945,)

In [187]:
viewer.layers[0].face_color

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       ...,
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [188]:
viewer.layers[0].face_color.shape

(945, 4)

In [168]:
for layer in viewer.layers:
    layer.scale = (0.27, 0.13, 0.13)

In [180]:
plt.close('all')

In [165]:
points_2

array([-488.89199829, -488.89199829, 1667.45996094, ..., -379.1000061 ,
       -379.1000061 , 1609.7800293 ])

Traceback (most recent call last):
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/app/backends/_qt.py", line 963, in paintGL
    self._vispy_canvas.events.draw(region=None)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/napari/_vispy/canvas.py", line 655, in on_draw
    layer._update_draw(
  File "/home/brandon/anaconda3/envs/zebrafish-ms2-paper/lib/python3.12/site-packages/napari/layers/points/points.py"

In [150]:
df.get('Cell Position Z')

0      1122.650024
1      1122.540039
2      1124.319946
3      1124.609985
4      1125.030029
          ...     
891    1134.930054
892    1134.540039
893    1135.339966
894    1134.930054
895    1134.530029
Name: Cell Position Z, Length: 896, dtype: float64

In [145]:
type(points_1)

NoneType

In [123]:
plt.tight_layout()

In [94]:
colors

{'green': '#7AA974',
 'light_green': '#BFD598',
 'pale_green': '#DCECCB',
 'yellow': '#EAC264',
 'light_yellow': '#F3DAA9',
 'pale_yellow': '#FFEDCE',
 'blue': '#738FC1',
 'light_blue': '#A9BFE3',
 'pale_blue': '#C9D7EE',
 'red': '#D56C55',
 'light_red': '#E8B19D',
 'pale_red': '#F1D4C9',
 'purple': '#AB85AC',
 'light_purple': '#D4C2D9',
 'dark_green': '#7E9D90',
 'dark_brown': '#905426'}

In [47]:
k_on, k_off, r = res.x#initial_param_guess()
m = her1
out = np.log(hyp1f1(k_on / γ + m, k_on / γ + k_off / γ + m, -r / γ))

In [50]:
np.where(np.isinf(out))

(array([], dtype=int64),)

In [48]:
np.where(np.isnan(out))

(array([], dtype=int64),)

In [65]:
γ = 0.07#0.23
random_ids = np.random.randint(0, len(her1), 1000)
#res = fit_count_data(her1[random_ids], γ)
res = fit_count_data(this_her1, γ)

In [66]:
res

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 4121.0741435748505
        x: [ 1.396e-01  8.796e+00  6.265e+01]
      nit: 47
      jac: [ 9.095e-05 -3.565e-02  3.729e-03]
     nfev: 276
     njev: 69
 hess_inv: <3x3 LbfgsInvHessProduct with dtype=float64>

In [67]:
bins = np.arange(100)
counts, _ = np.histogram(this_her1, bins)
probs = counts / np.sum(counts)

# # fit to gamma
# mean_her1 = np.mean(her1)
# var_her1 = np.var(her1)
# a = mean_her1 ** 2 / var_her1
# scale = var_her1 / mean_her1

#x = np.linspace(0, 100, 1000)
# gamma_dist = gamma.pdf(x, a, scale=scale)

# two-state fit
k_on, k_off, r = res.x
fit_prob_dist = np.exp(two_state_log_likelihood(bins, k_on, k_off, r, γ))

plt.figure(figsize=(6.74, 5.1))
#plt.plot(bins[:-1], probs, 'k-', linewidth=4)
plt.plot(bins, fit_prob_dist, '-', color=colors['blue'], linewidth=2, alpha=1, label='Gamma fit')
plt.plot(bins[:-1], probs, 'ko', markersize=3, label='smFISH data')
plt.xlabel("$her1$ mRNA count", fontsize=fontsize)
plt.ylabel('probability density ', fontsize=fontsize)
plt.legend(fontsize=fontsize, facecolor='w')
ax = style_axes(plt.gca())
plt.tight_layout()

In [31]:
two_state_log_likelihood(bins, k_on, k_off, r, γ)

array([ -6.85885482,  -5.72040395,  -5.03577032,  -4.56366228,
        -4.21759471,  -3.95557937,  -3.75381866,  -3.59745141,
        -3.47655045,  -3.38414551,  -3.3151501 ,  -3.26573638,
        -3.23295029,  -3.21446369,  -3.20840878,  -3.21326399,
        -3.22777323,  -3.2508874 ,  -3.28172126,  -3.31952091,
        -3.36363901,  -3.41351559,  -3.46866297,  -3.52865384,
        -3.59311168,  -3.66170299,  -3.734131  ,  -3.81013045,
        -3.88946329,  -3.97191506,  -4.05729189,  -4.14541795,
        -4.23613326,  -4.32929184,  -4.42476015,  -4.5224157 ,
        -4.62214583,  -4.72384675,  -4.82742258,  -4.9327846 ,
        -5.03985053,  -5.14854394,  -5.2587937 ,  -5.37053349,
        -5.48370141,  -5.59823956,  -5.71409373,  -5.83121306,
        -5.9495498 ,  -6.06905907,  -6.1896986 ,  -6.31142854,
        -6.43421132,  -6.55801141,  -6.68279524,  -6.80853103,
        -6.93518864,  -7.06273952,  -7.19115656,  -7.32041398,
        -7.45048728,  -7.58135316,  -7.71298939,  -7.84

In [33]:
plt.figure()
plt.plot(bins, np.exp(two_state_log_likelihood(bins, k_on, k_off, r, γ)))

In [56]:
res.x

array([3.70132611e-01, 2.95167289e+01, 4.60891458e+02])